In [1]:
import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'

import os
from pathlib import Path
import sys

CWD = os.getcwd()
PARENT_DIR = os.path.dirname(CWD)
sys.path.append(PARENT_DIR)

In [ ]:
from src.data import TextDataset
from torch.utils.data import DataLoader

from src.modules import QLoRA
import torch.optim as optim

MODEL_NAME = "Qwen/Qwen3.5-2B"
EPOCH_SIZE = 100
EVAL_ITER = 2
LEARNING_RATE = 1e-3
ACCUMULATION_STEPS = 4

train_data = TextDataset(tokenizer_model=MODEL_NAME)
loader = DataLoader(train_data.data[train_data.data_sources[0]],
            batch_size=1,
            shuffle=True,
            collate_fn=lambda rows: rows[0],
        )
qwen_qlora = QLoRA(model_name=MODEL_NAME, rank=16, alpha=32, device=device)

c:\Users\taput\anaconda3\envs\chatbot-fraudster\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
params    = [p for n, p in qwen_qlora.named_parameters() if p.requires_grad ]
optimizer = optim.AdamW(params=params, weight_decay=1e-2, lr=LEARNING_RATE)

# Train

In [ ]:
for step, batch in enumerate(loader, start=1):
    if step > EPOCH_SIZE:
        break

    batch = {
        key: value.unsqueeze(0).to("cuda:0")
        for key, value in batch.items()
    }

    loss = qwen_qlora(**batch).loss

    if not torch.isfinite(loss):
        raise RuntimeError(f"Non-finite loss at step {step}: {loss.item()}")

    (loss / ACCUMULATION_STEPS).backward()
    if step % ACCUMULATION_STEPS == 0:
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    print(f"step {step}: loss={loss.item():.4f}")

In [ ]:
adapter_weights = {
    name: parameter.detach().cpu()
    for name, parameter in qwen_qlora.named_parameters()
    if name.endswith(("lora_a", "lora_b"))
}

torch.save(
    {
        "base_model": f"../models/Qwen/{MODEL_NAME}",
        "rank": 16,
        "alpha": 32,
        "weights": adapter_weights,
    },
    "qwen3.5-4b-toolace-lora-1.pt",
)

In [ ]:
# Check which parameters are trainable.
trainable = [
    name for name, p in qwen_qlora.named_parameters()
    if p.requires_grad
]

print("Trainable tensors:", len(trainable))
print("First few:", trainable[:10])

assert all(
    name.endswith(("lora_a", "lora_b"))
    for name in trainable
), "A non-LoRA parameter is trainable"

In [ ]:
# Save it to Huggingface transformer version:

In [ ]:
import json
from pathlib import Path

import torch
from safetensors.torch import save_file

SOURCE = Path("../models/qwen3.5-2b-toolace-lora-1.pt")
OUTPUT = Path("../models/Qwen/qwen3.5-2b-toolace-adapter-1")

checkpoint = torch.load(SOURCE, map_location="cpu", weights_only=True)
assert checkpoint["base_model"] == "Qwen/Qwen3.5-2B"

weights = checkpoint["weights"]
converted = {}
targets = set()

for name, tensor in weights.items():
    if name.endswith(".lora_a"):
        module_name = name.removesuffix(".lora_a")
        suffix = "lora_A.weight"
    elif name.endswith(".lora_b"):
        module_name = name.removesuffix(".lora_b")
        suffix = "lora_B.weight"
    else:
        raise ValueError(f"Unexpected checkpoint key: {name}")

    # PEFT-style key: base_model.model.<original module path>.<A or B>
    new_name = f"base_model.model.{module_name}.{suffix}"
    converted[new_name] = tensor.detach().cpu().contiguous()
    targets.add(module_name.split(".")[-1])

assert len(converted) == len(weights)
assert len(converted) % 2 == 0

OUTPUT.mkdir(exist_ok=True)
save_file(converted, OUTPUT / "adapter_model.safetensors")

config = {
    "peft_type": "LORA",
    "task_type": "CAUSAL_LM",
    "base_model_name_or_path": checkpoint["base_model"],
    "r": checkpoint["rank"],
    "lora_alpha": checkpoint["alpha"],
    "lora_dropout": 0.0,
    "bias": "none",
    "target_modules": sorted(targets),
    "inference_mode": True,
}

(OUTPUT / "adapter_config.json").write_text(
    json.dumps(config, indent=2), encoding="utf-8"
)
print(f"Exported {len(converted) // 2} LoRA pairs")
print("Targets:", sorted(targets))

In [ ]:
# vllm serve Qwen/Qwen3.5-2B --enable-auto-tool-choice --tool-call-parser qwen3_coder --gpu-memoru-utilization 0.85
# bfcl generate --model qwen35-2b-base-FC --test-category simple_python,multi_turn --include-input-log